# To Do & Org & Pkgs

In [ ]:
"""
Description:
    CHMC Implementation with AVF: FPI
    USE THE CORRECT ENVIRONMENT:  CHMC_FALL_2025
    YYYY-MM-DD

Author: John Gallagher
Created: 2025-09-28
Last Modified: 2026-02-11
Version: 1.0.0

Current Goals: 
To Do: Matched with CHMC_notes.txt
Feb 11, 2026
To Do
Working in cov_plot.ipynb
✓ Making some data from existing HMC sampler
    0) Versus number of iterations (after burn in)
        - 1e2, 1e3 5e3
        - maybe 20 dots 
    1)  Low dim for speed (4, 8, 16)
        - Stuck on 8, 16
        - Likely issue with jit not recompiling when updating parameters. 
    ✓  Make `large number' of chains (20 each)
- Metric for chains    
    
    ✓  Calc cov matrices
    ✓) Metric for a single chain cov matrix
    4) Metric on all chains' cov matrix
    5) Avg Metric on all chains' cov matrix

- Plot cov matrix comparison
    6) Plot avg metric
    7) Plot all chains
    6) Plot avg + shadow chain

"""

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
import time
from scipy.sparse import diags
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

# Integrator, Target, HMC Scan

In [ ]:
def gen_nnormal(dim = 2, precision_matrix=None, cov=None):
    
    class MultipleMatrices(Exception):
            pass
    if precision_matrix is not None and cov is not None:
        raise MultipleMatrices(
            "Please supply either a Precision Matrix or a Covariance Matrix"
        )
    if precision_matrix is None and cov is not None:
        precision_matrix = jnp.linalg.inv(cov)
    if precision_matrix is None and cov is None:
        precision_matrix = jnp.eye(dim)
        
    def nnormal(x):
        """n-Dim Gaussian target distribution."""
        return jnp.exp(-0.5 * (x @ precision_matrix @ x))
    return nnormal

def gen_perturb_precision(dim):
    prec = jnp.diag(jnp.ones(dim))
    prec += 0.05*jnp.diag(jnp.ones(dim-1), k=-1)
    prec += 0.05*jnp.diag(jnp.ones(dim-1), k=1)
    # L -= 0.1*jnp.tri(dim, k=-2)
    # prec_out = L @ L.T
    return prec

# May need determinant later. 

def qex(qp):
    """
    qex: q extracted from qp state vector
    """
    dim = len(qp) // 2
    return qp[:dim]


def pex(qp):
    """
    pex: p extracted from qp state vector
    """
    dim = len(qp) // 2
    return qp[dim:]


def J_sym(vec):
    """
    J is the symplectic Jacobian matrix for Hamiltonians where J = ([[0, I]])
    """
    dim = len(vec) // 2
    return jnp.concatenate([vec[dim:], -vec[:dim]])


def qJ_sym(vec):
    """
    updates q side of vector with qdot = p
    Returns
    array([p], [0])
    """
    dim = len(vec) // 2
    return jnp.concatenate([vec[dim:], jnp.zeros(dim)])


def pJ_sym(vec):
    """
    qp with p = -qdot only
    Returns
    array([p], [0])
    """
    dim = len(vec) // 2
    return jnp.concatenate([jnp.zeros(dim), -vec[:dim]])


def draw_p(qp, key):
    q = qex(qp)
    p = jax.random.normal(key, shape=(dim,))
    return jnp.concatenate([q, p]), None


def gen_leapfrog(gradH, tau, N):
    def leapfrog(qp):
        """
        Requires gradH, tau, N
        Leapfrog integrator
        Takes state vector qp, and integrates it according to hamiltonian Ham

        """

        def lf_step(carry_in, _):
            qp0 = carry_in
            qhalf_p0 = qp0 + 0.5 * tau * qJ_sym(gradH(qp0))
            qhalf_pout = qhalf_p0 + tau * pJ_sym(gradH(qhalf_p0))
            qp_out = qhalf_pout + 0.5 * tau * qJ_sym(gradH(qhalf_pout))
            return qp_out, _

        qp_final, _ = jax.lax.scan(lf_step, qp, xs=None, length=N)
        return qp_final

    return leapfrog

def gen_midpointFPI(gradH, tau, N, tol,maxIter, solve = jnp.linalg.solve):
    """
    Generates midpointFPI function with appropriate statics: 
    tau, tol, maxIter, 
    """
    def midpointFPI(qp, _):
        """
        FPI_mid integrator
        Requries qp:statevector, and gradH defined before hand

        y(i+1) = y(i) + tau * J_sym GradH( 0.5*(y(i)+y(i+1)))

        """
        x0 = qp

        def G(y):
            """
            G(y) = x0 + tau * J_sym GradH( 0.5*(x+y))
            """
            midpoint = 0.5 * (x0 + y)
            return x0 + tau * J_sym(gradH(midpoint))

        def F(y):
            return y - G(y)
        print(f"F(x0) shape: {F(x0).shape}")
        jacF = jax.jacobian(F)
        jac_test = jacF(x0)
        print(f"Jacobian shape: {jac_test.shape}")
        def newton_step(qp):
            qpout = x0 - solve(jacF(qp), F(qp))
            return qpout

        def cond(carry):
            i, qp = carry
            Fqp = F(qp)
            err = jnp.linalg.norm(Fqp)
            return (err > tol) & (i < maxIter)

        def body_step(carry):
            i, qp = carry
            return [i + 1, newton_step(qp)]

        _, qp_out = jax.lax.while_loop(cond, body_step, [0, qp])
        return qp_out, qp_out
    def midpointFPI_T(qp):
        qp_out, _ = jax.lax.scan(midpointFPI, qp, xs = None, length = N)
        return qp_out
    return midpointFPI_T

def accept(delta, key):
    alpha = jnp.minimum(1.0, jnp.exp(delta))
    u = jax.random.uniform(key, shape=())
    return u <= alpha


def gen_hmc_kernel(H, tau, N):
    gradH = jax.grad(H)
    integrator = gen_leapfrog(gradH, tau, N)

    def hmc_kernel(carry_in, key):
        carry, _, _ = carry_in
        qp0, _ = draw_p(carry, key)
        qp_star = integrator(qp0)
        deltaH =  H(qp_star) - H(qp0)  # -(final - init) = init -final
        is_accepted = accept(-deltaH, key)
        qp_out = jnp.where(is_accepted, qp_star, qp0)
        carry_out = [qp_out, deltaH, is_accepted]
        return carry_out, carry_out
    return hmc_kernel
def gen_chmc_kernel(H, tau, N, tol, maxIter, solve=jnp.linalg.solve):
    gradH = jax.grad(H)
    integrator = gen_midpointFPI(gradH, tau, N, tol, maxIter, solve=jnp.linalg.solve )

    def chmc_kernel(carry_in, key):
        carry, _, _ = carry_in
        qp0, _ = draw_p(carry, key)
        qp_star = integrator(qp0)
        deltaH = H(qp0) - H(qp_star)  # -(final - init) = init -final
        is_accepted = accept(deltaH, key)
        qp_out = jnp.where(is_accepted, qp_star, qp0)
        carry_out = [qp_out, deltaH, is_accepted]
        return carry_out, carry_out
    return chmc_kernel

def hmc_sampler(initial_sample, keys, H, tau, N):
    """
    inputs: initial_sample, keys, H, tau, T
    """
    hmc_kernel = gen_hmc_kernel(H, tau, N)
    _, samples = jax.lax.scan(hmc_kernel, initial_sample, xs=keys)
    return samples
def chmc_sampler(initial_sample, keys, H, tau, N, tol, maxIter, solve=jnp.linalg.solve):
    """
    inputs: initial_sample, keys, H, tau, N, tol, maxIter, solve=jnp.linalg.solve
    """
    chmc_kernel = gen_chmc_kernel(H, tau, N, tol, maxIter, solve=jnp.linalg.solve)
    _, samples = jax.lax.scan(chmc_kernel, initial_sample, xs=keys)
    return samples    

def gen_hamiltonian(Mass_inv, target):
    
    def hamiltonian(qp):
        q, p = qex(qp), pex(qp)
        return 0.5 * jnp.sum(p @ Mass_inv @ p) - jnp.log(target(q))
    return hamiltonian
def gen_hidim_hamiltonian(Mass_inv, Prec_mat):
    def hamiltonian(qp):
        q, p = qex(qp), pex(qp)
        return 0.5 * jnp.sum(p@Mass_inv@p) + 0.5* jnp.sum(q@Prec_mat@q)
    return hamiltonian
# def J_H(gH):
#     """Same operation as Symplectic Jacobian"""
#     return jnp.concatenate([gH[dim:], -gH[:dim]])

# Initialize Test Param & Test Scan

In [ ]:
key = jax.random.PRNGKey(1)

dim = 16
Mass_inv = jnp.eye(dim)
# pert_precision = gen_perturb_precision(dim)
# target = gen_nnormal(precision_matrix=pert_precision)
target_mat = gen_perturb_precision(dim)
hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)
# target = gen_nnormal(dim)
# hamiltonian = gen_hamiltonian(Mass_inv, target)


## FOR HIGH CONDITION PRECISION MATRIX
# hamiltonian = gen_hicond_hamiltonian(Mass_inv, pert_precision)
jit_H = jit(hamiltonian)
gradH = jax.grad(hamiltonian)

# jit_integrator = jax.jit(leapfrog)
# jit_integrator = jit(midpointFPI)

# Set parameters


initnum_samples = 1
mainnum_samples = 100000
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)
qp_init = jax.random.normal(key, shape=(2 * dim,))

# Structure of carry
# Single chain output of [sample, deltaH, accpeted]
# init_sample: [Array: sample, float: deltaH, bool: Accepted]
init_sample = [qp_init, 1, False]

tau = 0.05
T = 1
N = int(jnp.ceil(T/tau))
tol = 1e-3
maxIter = 2
# compile
start = time.time()
jhmc_sampler = jit(hmc_sampler, static_argnums=(2,3,4))
sample_hmc = jhmc_sampler(init_sample, keys_start, hamiltonian, tau, N)
end = time.time()
print("1st run:", end - start)
# main run
start = time.time()
sample_hmc = jhmc_sampler(init_sample, keys_main,  hamiltonian, tau, N)
end = time.time()
print(
    f"Main run:\n {mainnum_samples} runs: {end - start:.2f} \n 1 run:  {(end-start)/mainnum_samples}"
)
start = time.time()
jchmc_sampler = jit(chmc_sampler, static_argnums=(2,3,4,5,6))
sample_chmc = jchmc_sampler(init_sample, keys_start, hamiltonian, tau, N, tol, maxIter)
end = time.time()
print("CHMC 1st run:", end - start)
start = time.time()
sample_chmc = jchmc_sampler(init_sample, keys_main, hamiltonian, tau, N, tol, maxIter)
end = time.time()
print(
    f"CHMC Main run:\n {mainnum_samples: .3e} runs: {end - start:.2f} \n 1 run:  {(end-start)/mainnum_samples: .3e}"
)

# Plotting: 
## Generating Small Dim Data

- dims = 4, 
- chainlen = 1e3, 
- chains = 20
- For loop implementation
- Numpy for array

In [ ]:
lens = [1e2, 1e3, 5e3, 1e4]
numchains = 20
dims = [4, 8, 16]
chainring = {}

for length in lens:
    chainring[length] = {}
    for dim in dims:
        chainring[length][dim] = np.zeros((int(length), dim, numchains))
keychain = {}

for j,length in enumerate(lens):
    keychain[length] = jax.random.split(key, (int(lens[j]), numchains))

In [ ]:
numchains = 20
chainlen = 1000
chains = np.zeros((chainlen,dim,numchains))
keychains = jax.random.split(key, (lens, numchains))
print('chains shape: ', chains.shape ,'\n', 'keychains shape: ', keychains.shape)
for i in range(numchains):
    chains[:,:,i] = jchmc_sampler(init_sample, keychains[:,i,:], hamiltonian, tau, N, tol, maxIter)[0][:,:dim]

#### Test for cov

In [ ]:
X = sample_chmc[0][:,:4]
Xμ = jnp.mean(X, axis = 0)
print(f'X shape: {X.shape}, \n Xμ shape: {Xμ.shape}')
XΣ=(X - Xμ).T@(X-Xμ)
print(f'XΣ shape: {XΣ.shape}')

In [ ]:
def cov(X):
    Xμ = jnp.mean(X, axis = 0)
    n=X.shape[0]
    return (X - Xμ).T@(X-Xμ)/(n-1)

In [ ]:
dims = np.logspace(2,4,3,base=2, dtype=int)

In [ ]:
target_mats = {}
target_mats[4] = gen_perturb_precision(4)
target_mats[8] = gen_perturb_precision(8)
target_mats[16] = gen_perturb_precision(16)

In [ ]:
for dimension in dims:
    print(type(dimension))
    target_mat = target_mats[int(dimension)]
    print(target_mat.shape)

In [ ]:
rawdata = {}
for dimension in dims:
    target_mat = target_mats[int(dimension)]
    Mass_inv = jnp.eye(int(dimension))
    hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)
    gradH = jax.grad(hamiltonian)
    qp_init = jax.random.normal(key, shape=(2 * int(dimension),))
    init_sample = [qp_init, 1, False]
    for i in range(numchains):
        chains[:,:,i] = chmc_sampler(init_sample, keychains[:,i,:], hamiltonian, tau, N, tol, maxIter)[0][:,:dimension]
    rawdata[int(dimension)] = chains

In [ ]:
rawdata = {}
for dimension in dims[1:]:
    print(dimension)
    target_mat = target_mats[int(dimension)]
    print(f'target_mat: {target_mat.shape}')
    Mass_inv = jnp.eye(int(dimension))
    print(f'Mass_inv: {Mass_inv.shape}')
    hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)
    gradH = jax.grad(hamiltonian)
    qp_init = jax.random.normal(key, shape=(2 * int(dimension),))
    print(gradH(qp_init).shape)
    print(qp_init.shape)
    init_sample = [qp_init, 1, False]
    chmc_sampler(init_sample, keychains[:,i,:], hamiltonian, tau, N, tol, maxIter)
    for i in range(numchains):
        chains[:,:,i] = chmc_sampler(init_sample, keychains[:,i,:], hamiltonian, tau, N, tol, maxIter)[0][:,:dimension]
    rawdata[int(dimension)] = chains

## Gen Cov Matrices & Diag Norm
```cov``` has a dict structure with indices 

```cov [dim][chainnum]: covariance_matrix```

```[dim]``` is an ```int: {4, 8, 16}```

```[chainnum]``` is an ```int: [0:19]```

In [ ]:
covs = {}
for idx in rawdata.keys():
    idxdict = {}
    for chainnum in range(rawdata[idx].shape[-1]):
        idxdict[chainnum]=cov(rawdata[idx][:,:,chainnum])
    covs[idx] = idxdict
## cov

In [ ]:
def maxdiagdiff(X,Y):
    x = np.diag(X)
    y = np.diag(Y)
    return np.max(np.abs(x-y))

### Generate Diag Error Data Set

In [ ]:
np.diag(covs[8][0])

In [ ]:
import matplotlib.pyplot as plt

chmc_dims = np.logspace(2,12,21,base=2, dtype=int)
def alphaex(deltaH):
    return jnp.minimum(1., jnp.exp(deltaH))
valphaex = jax.vmap(alphaex)
alphas = alphaex(samples_deltaHs)
meanalphas = alphas.mean(axis=0)
# meanalphas.shape
# intersection = np.interp(dims,dims, meanalphas)
chmc_alphas = alphaex(chmc_samples_deltaHs)
chmc_meanalphas = chmc_alphas.mean(axis=0)
sixteens = np.logspace(2,12,11,base=2)
for onetau in range(len(tau_set)):
    plt.semilogx(dims, meanalphas[onetau,:], marker = '.', label = f'$\\tau  = {tau_set[onetau]:.5f}$', base = 10)
# plt.semilogx(sixteens, np.ones(len(sixteens))*0.965, base=16,marker = '*', color ='red')

plt.semilogx(chmc_dims, chmc_meanalphas, marker = '*', label = f'$\\tau = 0.2$ CHMC', base=16)
plt.grid(True, which="major", ls="-", color='gray', alpha=0.5)
plt.grid(True, which="minor", ls=":", color='lightgray', alpha=0.4)
plt.ylabel(r'Mean$(\alpha)$')
plt.xlabel('Dimension')
plt.legend()
plt.title('Mean Acceptance Rate vs Dimension')
# plt.savefig('figures/LF_accept(tau)_v_dim_FINAL_base16.png')